# 6.2 Alpha beta pruning: practice notebook

Get the minimax answer while reading far fewer leaves. Learn what α and β mean, follow the slide example prune by prune, and see why move ordering lets the same time search twice as deep.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### Alpha beta on the slide example

In [ ]:
# Alpha beta pruning on the example of the slides. A is MAX, B and C are MIN, D, E, F, G are MAX.
tree = [[[2, 3], [5, 9]],          # B has D = [2, 3] and E = [5, 9]
        [[0, 1], [7, 5]]]          # C has F = [0, 1] and G = [7, 5]
big = 1000                          # stands for infinity

def alphabeta(node, is_max, alpha, beta):
    if type(node) == int:
        print("read leaf", node)
        return node
    if is_max:
        value = -big
        for child in node:
            value = max(value, alphabeta(child, False, alpha, beta))
            alpha = max(alpha, value)
            if alpha >= beta:
                print("  prune at a MAX node: alpha", alpha, ">= beta", beta)
                break
        return value
    value = big
    for child in node:
        value = min(value, alphabeta(child, True, alpha, beta))
        beta = min(beta, value)
        if alpha >= beta:
            print("  prune at a MIN node: alpha", alpha, ">= beta", beta)
            break
    return value

print("A =", alphabeta(tree, True, -big, big))

### Slide order against worst order

In [ ]:
# The same values in two different orders. Alpha beta gives the same answer, but reads a different number of leaves.
big = 1000
count = [0]                         # a list, so the function can change it

def alphabeta(node, is_max, alpha, beta):
    if type(node) == int:
        count[0] = count[0] + 1
        return node
    value = -big if is_max else big
    for child in node:
        v = alphabeta(child, not is_max, alpha, beta)
        if is_max:
            value = max(value, v)
            alpha = max(alpha, value)
        else:
            value = min(value, v)
            beta = min(beta, value)
        if alpha >= beta:
            break
    return value

good = [[[2, 3], [5, 9]], [[0, 1], [7, 5]]]       # the slide order
bad = [[[5, 7], [0, 1]], [[5, 9], [2, 3]]]        # each node sees its worst child first
for name, tree in [("slide order", good), ("worst order", bad)]:
    count[0] = 0
    v = alphabeta(tree, True, -big, big)
    print(name, ": value", v, ", leaves read", count[0], "of 8")

## Your turn

### Task 1: Alpha beta on a nested list tree

Write `alphabeta(node, is_max, alpha, beta)`. A node is a number (a leaf) or a list of children. Return the same value as minimax, but stop looking at the remaining children of a node as soon as alpha ≥ beta. Use −1000 and 1000 for −∞ and +∞.

Hint: Copy minimax, add alpha and beta as inputs, update alpha at MAX nodes and beta at MIN nodes, and break when alpha >= beta.

In [ ]:
def alphabeta(node, is_max, alpha, beta):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('alphabeta([[[2, 3], [5, 9]], [[0, 1], [7, 5]]], True, -1000, 1000)', 3), ('alphabeta([[3, 12, 8], [2, 4, 6], [14, 5, 2]], True, -1000, 1000)', 3), ('alphabeta([[[2, 1, 5], [6, 2, 11], [8, 5, 1]], [[3, 2, 4], [10, 11, 12], [1, 2, 1]], [[2, 1, 3], [6, 1, 11], [2, 5, 1]]], True, -1000, 1000)', 5), ('alphabeta(7, True, -1000, 1000)', 7)])

### Task 2: Count the leaves alpha beta reads

Write `count_leaves(tree)`. Run alpha beta from a MAX root with −1000 and 1000 and return `[value, leaves]`, where leaves is how many leaf numbers were actually read.

Hint: Keep the counter in a list so the inner function can change it. Add 1 each time you return a leaf.

In [ ]:
def count_leaves(tree):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('count_leaves([[[2, 3], [5, 9]], [[0, 1], [7, 5]]])', [3, 5]), ('count_leaves([[[5, 7], [0, 1]], [[5, 9], [2, 3]]])', [3, 8]), ('count_leaves([[3, 12, 8], [2, 4, 6], [14, 5, 2]])', [3, 7]), ('count_leaves([[[2, 1, 5], [6, 2, 11], [8, 5, 1]], [[3, 2, 4], [10, 11, 12], [1, 2, 1]], [[2, 1, 3], [6, 1, 11], [2, 5, 1]]])', [5, 11])])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: Alpha beta on a nested list tree

The three trees are the slide alpha beta example, the AIMA figure 5.2 tree and the 27 leaf slide minimax tree.

In [ ]:
def alphabeta(node, is_max, alpha, beta):
    if type(node) == int:
        return node
    if is_max:
        value = -1000
        for child in node:
            value = max(value, alphabeta(child, False, alpha, beta))
            alpha = max(alpha, value)
            if alpha >= beta:
                break
        return value
    value = 1000
    for child in node:
        value = min(value, alphabeta(child, True, alpha, beta))
        beta = min(beta, value)
        if alpha >= beta:
            break
    return value

In [ ]:
check([('alphabeta([[[2, 3], [5, 9]], [[0, 1], [7, 5]]], True, -1000, 1000)', 3), ('alphabeta([[3, 12, 8], [2, 4, 6], [14, 5, 2]], True, -1000, 1000)', 3), ('alphabeta([[[2, 1, 5], [6, 2, 11], [8, 5, 1]], [[3, 2, 4], [10, 11, 12], [1, 2, 1]], [[2, 1, 3], [6, 1, 11], [2, 5, 1]]], True, -1000, 1000)', 5), ('alphabeta(7, True, -1000, 1000)', 7)])

### Solution to task 2: Count the leaves alpha beta reads

The second test is the slide tree in the worst order: nothing is pruned. The last test is the 27 leaf tree, which is already in the best order: 11 = 3² + 3 − 1.

In [ ]:
def count_leaves(tree):
    count = [0]

    def alphabeta(node, is_max, alpha, beta):
        if type(node) == int:
            count[0] = count[0] + 1
            return node
        value = -1000 if is_max else 1000
        for child in node:
            v = alphabeta(child, not is_max, alpha, beta)
            if is_max:
                value = max(value, v)
                alpha = max(alpha, value)
            else:
                value = min(value, v)
                beta = min(beta, value)
            if alpha >= beta:
                break
        return value

    value = alphabeta(tree, True, -1000, 1000)
    return [value, count[0]]

In [ ]:
check([('count_leaves([[[2, 3], [5, 9]], [[0, 1], [7, 5]]])', [3, 5]), ('count_leaves([[[5, 7], [0, 1]], [[5, 9], [2, 3]]])', [3, 8]), ('count_leaves([[3, 12, 8], [2, 4, 6], [14, 5, 2]])', [3, 7]), ('count_leaves([[[2, 1, 5], [6, 2, 11], [8, 5, 1]], [[3, 2, 4], [10, 11, 12], [1, 2, 1]], [[2, 1, 3], [6, 1, 11], [2, 5, 1]]])', [5, 11])])